In [4]:
"""
Q1.1 读取附件1并合并为长表
输出: data/hourly_long.parquet  (columns: i, h, p)
"""
import pandas as pd
from pathlib import Path

# Jupyter 环境不使用 __file__，直接定义项目根目录
PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
SRC = PROJECT_ROOT / "A题" / "附件1.xlsx"
OUT = PROJECT_ROOT / "Q1输出" / "data"
OUT.mkdir(parents=True, exist_ok=True)

xl = pd.ExcelFile(SRC, engine="openpyxl")
print("Sheets:", xl.sheet_names)

frames = []
for sn in xl.sheet_names:
    df = xl.parse(sn)
    # 规范列名
    df.columns = [c.strip().lower() for c in df.columns]
    assert set(df.columns) >= {"time", "per"}, f"{sn} columns: {df.columns}"
    # 提取编号 i
    i = int(sn.split("_")[1])
    df = df.rename(columns={"time": "h", "per": "p"})[["h", "p"]]
    df["i"] = i
    df["h"] = pd.to_datetime(df["h"])
    df["p"] = pd.to_numeric(df["p"], errors="coerce")
    frames.append(df)
    print(f"  A_{i}: rows={len(df)}, p>100 count={(df['p']>100).sum()}, "
          f"p<0 count={(df['p']<0).sum()}, NaN={df['p'].isna().sum()}, "
          f"range=[{df['p'].min():.2f}, {df['p'].max():.2f}]")

long = pd.concat(frames, ignore_index=True).sort_values(["i", "h"]).reset_index(drop=True)
print(f"\nTotal rows: {len(long)}, filters: {long['i'].nunique()}")
print(f"Time span: {long['h'].min()} .. {long['h'].max()}")

# 保存
long.to_parquet(OUT / "hourly_long.parquet", index=False) if False else None
long.to_csv(OUT / "hourly_long.csv", index=False)
print(f"Saved: {OUT/'hourly_long.csv'}")


Sheets: ['A_1', 'A_2', 'A_3', 'A_4', 'A_5', 'A_6', 'A_7', 'A_8', 'A_9', 'A_10']
  A_1: rows=10839, p>100 count=3700, p<0 count=0, NaN=457, range=[38.64, 140.37]
  A_2: rows=11125, p>100 count=76, p<0 count=0, NaN=406, range=[28.06, 121.72]
  A_3: rows=11921, p>100 count=3759, p<0 count=0, NaN=409, range=[51.50, 154.57]
  A_4: rows=11675, p>100 count=4645, p<0 count=0, NaN=445, range=[31.56, 143.75]
  A_5: rows=12162, p>100 count=5561, p<0 count=0, NaN=458, range=[34.53, 149.81]
  A_6: rows=11806, p>100 count=1203, p<0 count=0, NaN=462, range=[28.65, 155.92]
  A_7: rows=11976, p>100 count=2276, p<0 count=0, NaN=456, range=[39.41, 158.57]
  A_8: rows=11507, p>100 count=4325, p<0 count=0, NaN=418, range=[37.63, 165.15]
  A_9: rows=11223, p>100 count=1208, p<0 count=0, NaN=455, range=[44.19, 139.39]
  A_10: rows=10743, p>100 count=3821, p<0 count=0, NaN=379, range=[26.43, 180.00]

Total rows: 114977, filters: 10
Time span: 2024-04-03 01:00:05 .. 2026-04-11 18:00:00
Saved: d:\Users\32174\De

In [5]:

# 简要统计
stats = long.groupby("i")["p"].agg(["count", "min", "max", "mean", "median"]).round(2)
stats["nan_count"] = long.groupby("i")["p"].apply(lambda s: s.isna().sum())
print("\nPer-filter summary:")
print(stats)
stats.to_csv(OUT / "per_filter_hourly_stats.csv")



Per-filter summary:
    count    min     max   mean  median  nan_count
i                                                 
1   10382  38.64  140.37  91.51   95.08        457
2   10719  28.06  121.72  73.73   73.79        406
3   11512  51.50  154.57  93.64   93.44        409
4   11230  31.56  143.75  92.00   96.74        445
5   11704  34.53  149.81  93.14   99.25        458
6   11344  28.65  155.92  74.03   73.32        462
7   11520  39.41  158.57  89.90   91.94        456
8   11089  37.63  165.15  95.54   95.03        418
9   10768  44.19  139.39  82.32   81.16        455
10  10364  26.43  180.00  94.85   94.50        379
